# Description

In this notebook, I will explore the benchmark Human Eval and using GPT4 to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
import load_dotenv
from openai import OpenAI

# 1. Load data

In [2]:
PATH_CSV_DATA = "data/human_eval.csv"

In [3]:
load_dotenv.load_dotenv()

# OPEN_AI_API = os.getenv("OPEN_AI_API")
OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")

client = OpenAI(api_key=OPEN_AI_API)

API key loaded successfully


In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
110,HumanEval/110,"\ndef exchange(lst1, lst2):\n """"""In this pr...",odd = 0\n even = 0\n for i in lst1:\...,def check(candidate):\n\n # Check some simp...,exchange


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def file_name_check(file_name):
    """Create a function which takes a string representing a file's name, and returns
    'Yes' if the the file's name is valid, and returns 'No' otherwise.
    A file's name is considered to be valid if and only if all the following conditions 
    are met:
    - There should not be more than three digits ('0'-'9') in the file's name.
    - The file's name contains exactly one dot '.'
    - The substring before the dot should not be empty, and it starts with a letter from 
    the latin alphapet ('a'-'z' and 'A'-'Z').
    - The substring after the dot should be one of these: ['txt', 'exe', 'dll']
    Examples:
    file_name_check("example.txt") # => 'Yes'
    file_name_check("1example.dll") # => 'No' (the name should start with a latin alphapet letter)
    """

Test Case:
def check(candidate):

    # Check some simple cases
    assert candidate("example.txt") == 'Yes'
    assert candidate("1example.dll") == 'No'
    assert candidate('

# 2. Using GPT4 to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

def extract_reasoning(llm_text):
    # 1) Grab text between <reasoning>...</reasoning>
    m = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <reasoning> block found")
    reasoning = m.group(1).strip()
    return reasoning

In [7]:
constraints = """
Output only a complete and valid Python code for this function. Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

COT = """
Before giving the final code, you MUST think step-by-step.
1. what the function must do  
2. possible edge cases  
3. the algorithm you will implement  
4. why this algorithm is correct  
5. then produce the final code

After reasoning, output the final answer strictly in this format:
<reasoning>
... your reasoning ...
</reasoning>
"""


input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
{COT}.
with the following constraints:\n{constraints}
"""

print("Input prompt to GPT-4:")
print(input_prompt)

Input prompt to GPT-4:
write a complete python function
based on the following description:

def file_name_check(file_name):
    """Create a function which takes a string representing a file's name, and returns
    'Yes' if the the file's name is valid, and returns 'No' otherwise.
    A file's name is considered to be valid if and only if all the following conditions 
    are met:
    - There should not be more than three digits ('0'-'9') in the file's name.
    - The file's name contains exactly one dot '.'
    - The substring before the dot should not be empty, and it starts with a letter from 
    the latin alphapet ('a'-'z' and 'A'-'Z').
    - The substring after the dot should be one of these: ['txt', 'exe', 'dll']
    Examples:
    file_name_check("example.txt") # => 'Yes'
    file_name_check("1example.dll") # => 'No' (the name should start with a latin alphapet letter)
    """
.


Before giving the final code, you MUST think step-by-step.
1. what the function must do  
2. possib

In [8]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o" 
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
<reasoning>
1. The function must check if a file name is valid based on specific rules:
   - No more than three digits in the name.
   - Exactly one dot separating the name and extension.
   - The name before the dot must start with a letter and can't be empty.
   - The extension must be among 'txt', 'exe', or 'dll'.

2. Possible edge cases include:
   - File names with multiple dots.
   - Names starting with numbers or special characters.
   - Empty strings or strings without a dot.
   - Strings with valid names but invalid extensions.

3. The algorithm:
   - Count digits in the file name; return 'No' if more than 3.
   - Split the string by the dot; it must result in exactly two parts.
   - Check that the first part starts with a letter and isn't empty.
   - Check that the second part is a valid extension ('txt', 'exe', 'dll').

4. This algorithm systematically verifies each condition outlined for a valid file name, ensuring accuracy in all edge cases.

By follo

We can extract the complete code

In [9]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def file_name_check(file_name):
    """Create a function which takes a string representing a file's name, and returns
    'Yes' if the the file's name is valid, and returns 'No' otherwise."""
    
    # Condition 1: Check number of digits
    num_digits = sum(char.isdigit() for char in file_name)
    if num_digits > 3:
        return 'No'
    
    # Condition 2: Split by '.' and check its validity
    parts = file_name.split('.')
    if len(parts) != 2:
        return 'No'
    
    name, extension = parts
    
    # Condition 3: Check the substring before the dot
    if not name or not name[0].isalpha():
        return 'No'
    
    # Condition 4: Check the extension validity
    if extension not in ['txt', 'exe', 'dll']:
        return 'No'
    
    return 'Yes'


In [10]:
reasoning_text = extract_reasoning(output)
print(f"The reasoning:\n")
print(reasoning_text)

The reasoning:

1. The function must check if a file name is valid based on specific rules:
   - No more than three digits in the name.
   - Exactly one dot separating the name and extension.
   - The name before the dot must start with a letter and can't be empty.
   - The extension must be among 'txt', 'exe', or 'dll'.

2. Possible edge cases include:
   - File names with multiple dots.
   - Names starting with numbers or special characters.
   - Empty strings or strings without a dot.
   - Strings with valid names but invalid extensions.

3. The algorithm:
   - Count digits in the file name; return 'No' if more than 3.
   - Split the string by the dot; it must result in exactly two parts.
   - Check that the first part starts with a letter and isn't empty.
   - Check that the second part is a valid extension ('txt', 'exe', 'dll').

4. This algorithm systematically verifies each condition outlined for a valid file name, ensuring accuracy in all edge cases.

By following these steps, 

## 2.2. Evaluate the generated code

In [11]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [12]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [13]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 26, 'passed': 26, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [14]:
list_df = []

for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx}/{df.shape[0]}")
    
    # 1. Prepare input prompt
    code_description = df.loc[idx, "prompt"]
    test_case = df.loc[idx, "test"]
    entry_point = df.loc[idx, "entry_point"]

    input_prompt = f"""write a complete python function
    based on the following description:\n{code_description}.\n
    {COT}.
    with the following constraints:\n{constraints}
    """

    # 2. Generate code with GPT-4o
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4o" 
        messages=[
            {"role": "system", "content": "You are an expert in Python."},
            {"role": "user", "content": input_prompt},
        ],
    )

    output = response.choices[0].message.content
    completed_code = extract_function(output)
    reasoning_text = extract_reasoning(output)
    
    # 3. Evaluate the generated code
    result = evaluate_asserts(completed_code, test_case, entry_point)
    total_asserts = result["total_asserts"]
    passed_asserts = result["passed"]
    percentage = result["percentage"]
    
    list_df.append({
        "description": code_description,
        "generated_code": completed_code,
        "test_case": test_case,
        "entry_point": entry_point,
        "total_asserts": total_asserts,
        "passed_asserts": passed_asserts,
        "percentage": percentage,
        "reasoning_text": reasoning_text,
    })

Processing idx=0/164
Processing idx=10/164
Processing idx=20/164
Processing idx=30/164
Processing idx=40/164
Processing idx=50/164
Processing idx=60/164
Processing idx=70/164
Processing idx=80/164
Processing idx=90/164
Processing idx=100/164
Processing idx=110/164
Processing idx=120/164
Processing idx=130/164
Processing idx=140/164
Processing idx=150/164
Processing idx=160/164


In [15]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 8)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage,reasoning_text
31,"\n\ndef is_prime(n):\n """"""Return true if a ...","import math\n\ndef is_prime(n):\n """"""Return...",\n\nMETADATA = {}\n\n\ndef check(candidate):\n...,is_prime,13,13,1.0,1. The function `is_prime(n)` must determine w...


In [16]:
# Save to CSV
output_df.to_csv("data/human_eval_generated_gpt4o_COT.csv", index=False)

## 3.1. Check generated code

In [17]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 94.95%


In [18]:
total_num_passed = output_df["passed_asserts"].sum()
print(f"Total number of passed asserts: {total_num_passed}")

Total number of passed asserts: 1138


## 3.2. Inspect incorrect 